In [ ]:
%run ./imports.py

In [ ]:
min_dev_hull_mainland_df_path = "data/min_dev_hull_mainland_df.pkl"
prefix_filtered_output = "data/campus_trace_prefixes_filtered_on_minrtt.pkl"
min_dev_pred_campus = "data/min_dev_pred_campus.pkl"
campus_predicted_owds = "data/campus_trace_predicted_owds.pkl"

In [ ]:
C_OPTICAL_FIBER_KM_PER_MS = (2/3) * (299792458 / 10**6)

### Optimal attacks

In [ ]:
df_campus_mindev = pd.read_pickle(min_dev_hull_mainland_df_path)
print(df_campus_mindev.shape[0])

In [ ]:
df_campus_mindev.head(n=1)

### Campus trace prefixes

In [ ]:
df_campus_prefixes = pd.read_pickle(prefix_filtered_output)
df_campus_prefixes = df_campus_prefixes[df_campus_prefixes["RTT_Count"] >= 10]
df_campus_prefixes = df_campus_prefixes.rename(columns={
    "Destination_Prefix": "prefix",
    "Geodesic_Distance_km": "distance_km",
    "Latitude": "clat",
    "Longitude": "clon",
    "minRTT_ms": "minrtt"
})
df_campus_prefixes = df_campus_prefixes.loc[:, ["prefix", "clat", "clon", "minrtt", "distance_km"]]
df_campus_prefixes = df_campus_prefixes.groupby(["prefix", "clat", "clon"], as_index=False).agg({
    'minrtt': 'min',
    'distance_km': 'min'
})
print(df_campus_prefixes.shape[0])

In [ ]:
df_campus_prefixes.head(n=1)

In [ ]:
df_campus_filtered = df_campus_prefixes[df_campus_prefixes["minrtt"] >= (2 * df_campus_prefixes["distance_km"]) / C_OPTICAL_FIBER_KM_PER_MS].copy()
print(df_campus_filtered.shape[0])

In [ ]:
df_campus_filtered.head(n=1)

In [ ]:
df_campus_bucket = df_campus_filtered[["distance_km", "minrtt"]].copy()
df_campus_bucket["distance_bin_km"] = df_campus_bucket["distance_km"].apply(lambda d: int(round(d / 200) * 200))
df_campus_bucket["minowd_bin_ms"] = df_campus_bucket["minrtt"].apply(lambda r: int(round(r/2)))
df_campus_bucket.head(n=1)

In [ ]:
df_campus_dist_bucket = df_campus_bucket.groupby("distance_bin_km", as_index=False).agg({"minowd_bin_ms": list})
df_campus_dist_bucket["minowd_cof_ms"] = df_campus_dist_bucket["distance_bin_km"].apply(lambda d: d / C_OPTICAL_FIBER_KM_PER_MS)
df_campus_dist_bucket["minowd_p05_ms"] = df_campus_dist_bucket["minowd_bin_ms"].apply(lambda r: np.percentile(r, 5))
df_campus_dist_bucket["minowd_p10_ms"] = df_campus_dist_bucket["minowd_bin_ms"].apply(lambda r: np.percentile(r, 10))
df_campus_dist_bucket["minowd_p25_ms"] = df_campus_dist_bucket["minowd_bin_ms"].apply(lambda r: np.percentile(r, 25))
df_campus_dist_bucket["minowd_p50_ms"] = df_campus_dist_bucket["minowd_bin_ms"].apply(lambda r: np.percentile(r, 50))
df_campus_dist_bucket["minowd_p75_ms"] = df_campus_dist_bucket["minowd_bin_ms"].apply(lambda r: np.percentile(r, 75))
df_campus_dist_bucket["minowd_p90_ms"] = df_campus_dist_bucket["minowd_bin_ms"].apply(lambda r: np.percentile(r, 90))
df_campus_dist_bucket["minowd_p95_ms"] = df_campus_dist_bucket["minowd_bin_ms"].apply(lambda r: np.percentile(r, 95))
df_campus_dist_bucket.head(n=3)

In [ ]:
distances = df_campus_dist_bucket["distance_bin_km"].tolist()
owd_cof = df_campus_dist_bucket["minowd_cof_ms"].tolist()
owd_p05 = df_campus_dist_bucket["minowd_p05_ms"].tolist()
owd_p10 = df_campus_dist_bucket["minowd_p10_ms"].tolist()
owd_p25 = df_campus_dist_bucket["minowd_p25_ms"].tolist()
owd_p50 = df_campus_dist_bucket["minowd_p50_ms"].tolist()
owd_p75 = df_campus_dist_bucket["minowd_p75_ms"].tolist()
owd_p90 = df_campus_dist_bucket["minowd_p90_ms"].tolist()
owd_p95 = df_campus_dist_bucket["minowd_p95_ms"].tolist()

In [ ]:
def predict_owds(dists, owds_measured):
    dists = np.array(dists).reshape(-1, 1)
    model = LinearRegression()
    model.fit(dists, owds_measured)
    print(f"Linear Regression Slope = {model.coef_[0]}, Intercept = {model.intercept_}")
    smooth_distances = np.array(range(20039)).reshape(-1, 1)
    owds_predicted   = model.predict(smooth_distances)
    return smooth_distances, owds_predicted

In [ ]:
smoothed_distances, owds_p05_predicted = predict_owds(distances, owd_p05)
smoothed_distances, owds_p10_predicted = predict_owds(distances, owd_p10)
smoothed_distances, owds_p25_predicted = predict_owds(distances, owd_p25)
smoothed_distances, owds_p50_predicted = predict_owds(distances, owd_p50)
smoothed_distances, owds_p75_predicted = predict_owds(distances, owd_p75)
smoothed_distances, owds_p90_predicted = predict_owds(distances, owd_p90)
smoothed_distances, owds_p95_predicted = predict_owds(distances, owd_p95)
smoothed_distances = [d[0] for d in smoothed_distances]

In [ ]:
def pred_to_dict(dists, owds):
    pred_dict = {}
    for d, o in zip(dists, owds):
        pred_dict[d] = o
    return pred_dict

pred_p05 = pred_to_dict(smoothed_distances, owds_p05_predicted)
pred_p10 = pred_to_dict(smoothed_distances, owds_p10_predicted)
pred_p25 = pred_to_dict(smoothed_distances, owds_p25_predicted)
pred_p50 = pred_to_dict(smoothed_distances, owds_p50_predicted)
pred_p75 = pred_to_dict(smoothed_distances, owds_p75_predicted)
pred_p90 = pred_to_dict(smoothed_distances, owds_p90_predicted)
pred_p95 = pred_to_dict(smoothed_distances, owds_p95_predicted)

In [ ]:
pu.lineplots([smoothed_distances] * 7,
             [owds_p05_predicted, owds_p10_predicted, owds_p25_predicted, owds_p50_predicted,
              owds_p75_predicted, owds_p90_predicted, owds_p95_predicted],
            {"curvelabels": [f"p{i}" for i in [5,10,25,50,75,90,95]]})

In [ ]:
with open(campus_predicted_owds, "wb") as fp:
    pickle.dump({
        "p05": pred_p05,
        "p10": pred_p10,
        "p25": pred_p25,
        "p50": pred_p50,
        "p75": pred_p75,
        "p90": pred_p90,
        "p95": pred_p95
    }, fp)

In [ ]:
scatter_dists = []
scatter_owds  = []
for d, ol in zip(df_campus_dist_bucket["distance_bin_km"].tolist(), df_campus_dist_bucket["minowd_bin_ms"].tolist()):
    for o in random.sample(ol, min(50, len(ol))):
        scatter_dists.append(d)
        scatter_owds.append(o)

In [ ]:
colors = list(sns.color_palette("bright", n_colors=3))
_, ax, props = pu.setup_plot({"figsize": (8, 6), "xlabel": "Geodesic Distance (km)", "ylabel": "Minimum OWD (ms)", "ylim": (-10, 310)})
ax.scatter(scatter_dists, scatter_owds,
           marker="x", color="lightgrey", label="Campus Measurement", s=props["markersize"])
ax.plot(smoothed_distances, [d / C_OPTICAL_FIBER_KM_PER_MS for d in smoothed_distances],
        color="darkgrey", label="Speed of Light", linewidth=props["linewidth"])
ax.plot(smoothed_distances, owds_p25_predicted, color=colors[2], label="p25 Regression Line", linewidth=props["linewidth"])
ax.plot(smoothed_distances, owds_p75_predicted, color=colors[1], label="p75 Regression Line", linewidth=props["linewidth"])
ax.legend(framealpha=1)
props["plot_path"] = "plots/linreg_campus.pdf"
ax = pu.add_properties(ax, props)
pu.close_plot(ax, props)

### Coverage: Campus

In [ ]:
df_mindev = pd.read_pickle(min_dev_hull_mainland_df_path)
df_mindev.head(n=1)

In [ ]:
df_mindev_pred_campus = df_mindev[["Country1", "Country2", "Dist_SD_km", "Dist_SA_km", "Dist_DA_km"]].copy()

df_mindev_pred_campus["PreAttack_p95_ms"] = df_mindev_pred_campus.apply(lambda row:
                                                           2 * pred_p95[int(round(row["Dist_SD_km"]))]
                                                          , axis=1)
df_mindev_pred_campus["PreAttack_p90_ms"] = df_mindev_pred_campus.apply(lambda row:
                                                           2 * pred_p90[int(round(row["Dist_SD_km"]))]
                                                          , axis=1)
df_mindev_pred_campus["PreAttack_p75_ms"] = df_mindev_pred_campus.apply(lambda row:
                                                           2 * pred_p75[int(round(row["Dist_SD_km"]))]
                                                          , axis=1)
df_mindev_pred_campus["PreAttack_p50_ms"] = df_mindev_pred_campus.apply(lambda row:
                                                           2 * pred_p50[int(round(row["Dist_SD_km"]))]
                                                          , axis=1)

df_mindev_pred_campus["PostAttack_p05_ms"] = df_mindev_pred_campus.apply(lambda row:
                                                              pred_p05[int(round(row["Dist_SD_km"]))]
                                                            + pred_p05[int(round(row["Dist_SA_km"]))]
                                                            + pred_p05[int(round(row["Dist_DA_km"]))]
                                                           , axis=1)
df_mindev_pred_campus["PostAttack_p10_ms"] = df_mindev_pred_campus.apply(lambda row:
                                                              pred_p10[int(round(row["Dist_SD_km"]))]
                                                            + pred_p10[int(round(row["Dist_SA_km"]))]
                                                            + pred_p10[int(round(row["Dist_DA_km"]))]
                                                           , axis=1)
df_mindev_pred_campus["PostAttack_p25_ms"] = df_mindev_pred_campus.apply(lambda row:
                                                              pred_p25[int(round(row["Dist_SD_km"]))]
                                                            + pred_p25[int(round(row["Dist_SA_km"]))]
                                                            + pred_p25[int(round(row["Dist_DA_km"]))]
                                                           , axis=1)
df_mindev_pred_campus["PostAttack_p50_ms"] = df_mindev_pred_campus.apply(lambda row:
                                                              pred_p50[int(round(row["Dist_SD_km"]))]
                                                            + pred_p50[int(round(row["Dist_SA_km"]))]
                                                            + pred_p50[int(round(row["Dist_DA_km"]))]
                                                           , axis=1)

df_mindev_pred_campus["MinDev_p95_p05"] = df_mindev_pred_campus.apply(lambda row: max(row["PostAttack_p05_ms"] - row["PreAttack_p95_ms"], 0), axis=1)
df_mindev_pred_campus["MinDev_p90_p10"] = df_mindev_pred_campus.apply(lambda row: max(row["PostAttack_p10_ms"] - row["PreAttack_p90_ms"], 0), axis=1)
df_mindev_pred_campus["MinDev_p75_p25"] = df_mindev_pred_campus.apply(lambda row: max(row["PostAttack_p25_ms"] - row["PreAttack_p75_ms"], 0), axis=1)
df_mindev_pred_campus["MinDev_p50_p50"] = df_mindev_pred_campus.apply(lambda row: max(row["PostAttack_p50_ms"] - row["PreAttack_p50_ms"], 0), axis=1)

df_mindev_pred_campus.head(n=1)

In [ ]:
df_mindev_pred_campus.to_pickle(min_dev_pred_campus)

In [ ]:
sns_colors = list(sns.color_palette("bright"))

In [ ]:
countries = sorted(df_mindev_pred_campus["Country1"].unique())
preattacks = df_mindev_pred_campus["PreAttack_p75_ms"].tolist()
postattacks = df_mindev_pred_campus["PostAttack_p25_ms"].tolist()

x, y = [], []
for p in range(1, 101, 1):
    preattack_at_p = np.percentile(preattacks, p)
    defense_count = sum(1 for postattack in postattacks if postattack > preattack_at_p)
    defense_pct = defense_count * 100.0 / len(postattacks)
    x.append(p)
    y.append(defense_pct)

# for i, j in zip(x, y):
#     if round(j) >= 80 and round(j) <= 95:
#         print(f"{round(j, 1)}% attacks > {i}%ile pre-attack RTT")

pu.lineplot(x, y, {
        "figsize": (8, 6), "colors": [sns_colors[3]],
        # "ylim": (48, 102),
        "xlabel": "Pre-Attack MinRTT (%ile)",
        "ylabel": "Attack Scenarios (%)",
        "plot_path": "plots/campus_cvg_prep75_postp25.pdf"
})

In [ ]:
X, Y, Z = [], [], []
xticks_pos = []
yticks_pos = []
x_25, y_25 = [], []

for i, x in enumerate(range(0, 5001, 10)):
    for j, y in enumerate(range(0, 16001, 10)):
        if x > 0:
            pre  = 2 * pred_p75[x]
            post = pred_p25[x] + (2 * pred_p25[y])
            z = max(post - pre, 0)
            if round(z) == 25:
                x_25.append(x)
                y_25.append(y)
        
            X.append(x)
            Y.append(y)
            Z.append(z)

        if x % 1000 == 0:
            xticks_pos.append(i)
        if y % 4000 == 0:
            yticks_pos.append(j)

xticks_pos = sorted(list(set(xticks_pos)))
yticks_pos = sorted(list(set(yticks_pos)))

reload(pu)
pu.heatmap(X, Y, Z,
           {
                "figsize": (8, 6), "palette": "Reds", "bounds": list(range(0, 151, 25)),
                "label": "$\\tau_{mid} - \\tau_{pre}$ (ms)",
                "xlabel": "$\delta(S,D)$ ($\\times10^3$ km)",
                "ylabel": "$\\frac{\delta(S,A) + \delta(D,A)}{2}$ ($\\times10^3$ km)",
                "xticks": (xticks_pos, list(range(6))),
                "yticks": (yticks_pos, list(range(0, 17, 4))),
                "plot_path": "plots/dist_vs_dev_heatmap_campus.png"
            })

In [ ]:
X, Y, Z = [], [], []
xticks_pos = []
yticks_pos = []
x_25, y_25 = [], []

for i, x in enumerate(range(0, 5001, 10)):
    for j, y in enumerate(range(0, 16001, 10)):
        if x > 0:
            pre  = 2 * pred_p75[x]
            post = pred_p25[x] + (2 * pred_p25[y])
            z = max(post - pre, 0)
            if round(z) == 25:
                x_25.append(x)
                y_25.append(y)
        
            X.append(x)
            Y.append(y)
            Z.append(z)

        if x % 1000 == 0:
            xticks_pos.append(i)
        if y % 4000 == 0:
            yticks_pos.append(j)

xticks_pos = sorted(list(set(xticks_pos)))
yticks_pos = sorted(list(set(yticks_pos)))

reload(pu)
pu.heatmap(X, Y, Z,
           {
                "figsize": (8, 6), "palette": "Reds", "bounds": list(range(0, 151, 25)),
                "label": "Min. Deviation (ms)",
                "xlabel": "$\delta(S,D)$ ($\\times10^3$ km)",
                "ylabel": "$\\frac{\delta(S,A) + \delta(D,A)}{2}$ ($\\times10^3$ km)",
                "xticks": (xticks_pos, list(range(6))),
                "yticks": (yticks_pos, list(range(0, 17, 4)))
            })